# Tugas P10 — Segmentasi Polip pada Citra Kolonoskopi

**Kecerdasan Buatan (BM25-41204) · Teknik Biomedis ITERA** — Martin C.T. Manullang
Petunjuk lengkap tugas: **https://mctm.web.id/ai-bm/tugas**

Di kelas kita melatih U-Net untuk mewarnai **paru-paru** pada X-ray. Sekarang tugasnya lebih menantang: menemukan **polip** (benjolan di dinding usus besar) pada foto **kolonoskopi** berwarna. Polip bisa kecil, besar, datar, atau menonjol — jauh lebih sulit daripada paru.

### Pertanyaan yang harus kelompokmu jawab
> **Augmentasi apa yang membantu U-Net menemukan polip, dan augmentasi apa yang tidak? Kenapa?**

### Kamu TIDAK perlu menulis PyTorch dari nol
Notebook ini sudah lengkap dan bisa langsung dijalankan. Tugasmu:
1. Jalankan notebook apa adanya sekali (mode cepat) untuk memastikan semuanya jalan.
2. Ubah **hanya** sel yang bertanda **✏️ UBAH DI SINI**.
3. Jalankan ulang, lalu tulis temuanmu di `README.md` repo kelompok.

Semua gambar & tabel hasil otomatis tersimpan di folder `hasil/` — tinggal ditampilkan di README.

## Cara memakai notebook ini

- Jalankan sel **berurutan dari atas ke bawah** dengan **Shift + Enter**. Kalau bingung atau error, jalankan ulang semuanya dari atas (*Run All*).
- Sel bertanda **✏️ UBAH DI SINI** = boleh (dan harus) kamu ubah.
- Sel lain = **jangan diubah dulu**. Baca komentarnya (tulisan setelah `#`) untuk memahami isinya.
- Butuh GPU? Di Google Colab: **Runtime → Change runtime type → T4 GPU**. Di laptop tanpa GPU tetap jalan, hanya lebih lama.

---
## Bagian 1 — Menyiapkan alat

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.transforms import v2       # alat augmentasi
from torchvision import tv_tensors          # penanda: ini citra, ini mask

# Pilih "mesin": GPU NVIDIA (CUDA) → GPU Mac (MPS) → CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

FOLDER_HASIL = Path("hasil")
FOLDER_HASIL.mkdir(exist_ok=True)           # semua gambar & tabel disimpan di sini

print("PyTorch versi", torch.__version__, "| dilatih memakai:", device)

---
## Bagian 2 — ✏️ UBAH DI SINI: pengaturan eksperimen

| Pengaturan | Artinya |
|---|---|
| `CEPAT` | `True` = uji coba kilat (sedikit epoch, 1 ulangan) untuk memastikan kode jalan. **Hasil akhir untuk README wajib `False`.** |
| `JUMLAH_CITRA_LATIH` | Berapa citra yang dipakai untuk belajar (maks. 800). Sedikit data = augmentasi biasanya lebih terasa. |
| `JUMLAH_EPOCH` | Berapa kali model melihat seluruh data latih. |
| `JUMLAH_ULANGAN` | Setiap skenario diulang dengan citra latih & titik awal model yang berbeda, lalu dirata-rata. Minimal 3. |

In [ ]:
# ✏️ UBAH DI SINI
CEPAT = True                 # ganti False untuk hasil akhir
JUMLAH_CITRA_LATIH = 200
JUMLAH_EPOCH = 40
JUMLAH_ULANGAN = 3

if CEPAT:
    JUMLAH_EPOCH, JUMLAH_ULANGAN = 3, 1
    print("⚠️  MODE CEPAT — hanya untuk uji coba, JANGAN dipakai di README.")
print(f"Citra latih: {JUMLAH_CITRA_LATIH} | epoch: {JUMLAH_EPOCH} | ulangan: {JUMLAH_ULANGAN}")

---
## Bagian 3 — Mengunduh & menyiapkan data

**Kvasir-SEG** (Jha dkk., 2020): 1000 foto kolonoskopi dari rumah sakit di Norwegia, masing-masing dengan **mask** polip yang digambar dokter. Dataset (±160 MB) diunduh sekali ke folder `data/`, lalu diperkecil menjadi 128 × 128 piksel supaya pelatihan ringan.

> Folder `data/` **tidak** ikut di-commit (sudah diatur di `.gitignore`). Jangan pernah meng-commit dataset ke GitHub.

In [ ]:
UKURAN = 128
file_cache = Path("data/kvasir_128.npz")

if not file_cache.exists():
    try:
        import kagglehub
    except ImportError:
        %pip install -q kagglehub
        import kagglehub
    folder = Path(kagglehub.dataset_download("debeshjha1/kvasirseg", output_dir="data/kvasir-seg"))
    daftar = sorted((folder / "Kvasir-SEG" / "Kvasir-SEG" / "images").glob("*.jpg"))
    print("Memperkecil", len(daftar), "citra ...")
    citra = [np.asarray(Image.open(f).convert("RGB").resize((UKURAN, UKURAN), Image.Resampling.BILINEAR)) for f in daftar]
    mask = [np.asarray(Image.open(str(f).replace("images", "masks")).convert("L").resize((UKURAN, UKURAN), Image.Resampling.NEAREST)) > 127
            for f in daftar]
    np.savez_compressed(file_cache, citra=np.stack(citra), mask=np.stack(mask).astype(np.uint8))

data = np.load(file_cache)
# Citra berwarna → 3 kanal (merah, hijau, biru), skala 0..1. Bentuk: (jumlah, 3, 128, 128)
X = torch.tensor(data["citra"]).permute(0, 3, 1, 2).float() / 255
Y = torch.tensor(data["mask"]).float().unsqueeze(1)          # (jumlah, 1, 128, 128): 1 = polip

# Pembagian TETAP (jangan diubah): 200 citra uji, sisanya 800 untuk dipilih sebagai data latih
urutan = torch.randperm(len(X), generator=torch.Generator().manual_seed(0))
indeks_uji, kumpulan_latih = urutan[:200], urutan[200:]
print(f"Total {len(X)} citra | uji: {len(indeks_uji)} | kumpulan latih: {len(kumpulan_latih)}")
print(f"Rata-rata luas polip: {Y.mean():.1%} dari citra")

In [ ]:
def tampilkan_overlay(ax, citra, mask, warna=(0, 1, 0.4), judul=""):
    # citra: (3, H, W) tensor; mask: (H, W) 0/1
    ax.imshow(citra.permute(1, 2, 0).numpy())
    lapisan = np.zeros((*mask.shape, 4))
    lapisan[np.asarray(mask) == 1] = (*warna, 0.45)
    ax.imshow(lapisan)
    ax.set_title(judul, fontsize=10)
    ax.axis("off")


fig, axes = plt.subplots(2, 5, figsize=(16, 6.8))
for kolom, i in enumerate(indeks_uji[:5]):
    axes[0, kolom].imshow(X[i].permute(1, 2, 0).numpy()); axes[0, kolom].axis("off")
    axes[0, kolom].set_title(f"Citra #{i.item()}", fontsize=10)
    tampilkan_overlay(axes[1, kolom], X[i], Y[i, 0], judul=f"Mask polip ({Y[i].mean():.0%} citra)")
plt.suptitle("Contoh data Kvasir-SEG")
plt.tight_layout()
plt.savefig(FOLDER_HASIL / "contoh_data.png", dpi=120)
plt.show()

---
## Bagian 4 — ✏️ UBAH DI SINI: skenario augmentasi

Inilah inti tugasmu. Di bawah ada **katalog augmentasi** dari `torchvision.transforms.v2`. Untuk setiap augmentasi, pikirkan pertanyaan klinisnya: *apakah variasi ini mungkin terjadi pada foto kolonoskopi sungguhan?*

| Augmentasi | Kode | Pertanyaan untuk kelompokmu |
|---|---|---|
| Flip kiri-kanan | `v2.RandomHorizontalFlip()` | Pada X-ray, flip membuat jantung pindah sisi. Bagaimana pada kolonoskopi? |
| Flip atas-bawah | `v2.RandomVerticalFlip()` | Apakah kamera kolonoskopi punya arah "atas" yang pasti? |
| Rotasi | `v2.RandomRotation(degrees=180)` | Seberapa bebas kamera berputar di dalam usus? |
| Geser & perbesar | `v2.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.8, 1.2))` | Polip bisa difoto dari dekat atau jauh? |
| Kecerahan & kontras | `v2.ColorJitter(brightness=0.3, contrast=0.3)` | Apakah lampu endoskop selalu sama terangnya? |
| Saturasi | `v2.ColorJitter(saturation=0.3)` | Apa jadinya kalau warna lebih pucat/pekat? |
| Hue (rona warna) | `v2.ColorJitter(hue=0.1)` | Warna kemerahan mukosa adalah petunjuk penting. Aman diubah? |
| Blur | `v2.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))` | Kamera bergerak → foto kabur. Realistis? |
| Deformasi elastis | `v2.ElasticTransform(alpha=30.0)` | Dinding usus lentur. Apakah ini meniru bentuk polip yang berbeda-beda? |

**Cara membuat skenario:** gabungkan beberapa augmentasi dengan `v2.Compose([...])`, beri nama, lalu masukkan ke `SKENARIO`.

**Wajib:** pertahankan `"Tanpa augmentasi"` sebagai pembanding, lalu buat **minimal 2 skenario augmentasi**. Tulis **hipotesis & alasan klinis** tiap skenario di README **sebelum** menjalankan eksperimen.

In [ ]:
# ✏️ UBAH DI SINI
SKENARIO = {
    "Tanpa augmentasi": None,            # pembanding — JANGAN dihapus

    # Contoh skenario. Ganti/ubah/tambah sesuai hipotesis kelompokmu:
    "Flip": v2.Compose([
        v2.RandomHorizontalFlip(),
        v2.RandomVerticalFlip(),
    ]),
    "Kecerahan & kontras": v2.ColorJitter(brightness=0.3, contrast=0.3),
}

print("Skenario yang akan dijalankan:", list(SKENARIO))

### Cek augmentasimu secara visual
Sebelum melatih, **selalu lihat** hasil augmentasi. Pastikan mask (hijau) tetap menempel pada polip dan citranya masih masuk akal sebagai foto kolonoskopi.

In [ ]:
def terapkan_augmentasi(augmentasi, citra, jawaban):
    # citra & mask diubah BERSAMAAN; tv_tensors.Mask memastikan mask ikut diputar/digeser
    hasil = [augmentasi(tv_tensors.Image(c), tv_tensors.Mask(m)) for c, m in zip(citra, jawaban)]
    return torch.stack([c for c, _ in hasil]), torch.stack([m for _, m in hasil])


contoh = indeks_uji[0]
for nama, augmentasi in SKENARIO.items():
    if augmentasi is None:
        continue
    fig, axes = plt.subplots(1, 6, figsize=(17, 3.3))
    tampilkan_overlay(axes[0], X[contoh], Y[contoh, 0], judul="Asli")
    for ax in axes[1:]:
        c, m = terapkan_augmentasi(augmentasi, X[contoh : contoh + 1], Y[contoh : contoh + 1])
        tampilkan_overlay(ax, c[0].clamp(0, 1), m[0, 0], judul=nama)
    plt.tight_layout()
    nama_file = "augmentasi_" + nama.lower().replace(" ", "_").replace("&", "dan") + ".png"
    plt.savefig(FOLDER_HASIL / nama_file, dpi=110)
    plt.show()

---
## Bagian 5 — Model U-Net & cara melatih (tidak perlu diubah)

U-Net yang sama dengan di kelas. Bedanya hanya satu: masukannya **3 kanal** (citra berwarna), bukan 1.

In [ ]:
def blok_konvolusi(kanal_masuk, kanal_keluar):
    return nn.Sequential(
        nn.Conv2d(kanal_masuk, kanal_keluar, 3, padding=1), nn.BatchNorm2d(kanal_keluar), nn.ReLU(),
        nn.Conv2d(kanal_keluar, kanal_keluar, 3, padding=1), nn.BatchNorm2d(kanal_keluar), nn.ReLU(),
    )


class UNetMini(nn.Module):
    def __init__(self, k=16):
        super().__init__()
        self.ringkas = nn.MaxPool2d(2)
        self.enc1, self.enc2, self.enc3 = blok_konvolusi(3, k), blok_konvolusi(k, 2 * k), blok_konvolusi(2 * k, 4 * k)
        self.dasar = blok_konvolusi(4 * k, 8 * k)
        self.perbesar3, self.dec3 = nn.ConvTranspose2d(8 * k, 4 * k, 2, stride=2), blok_konvolusi(8 * k, 4 * k)
        self.perbesar2, self.dec2 = nn.ConvTranspose2d(4 * k, 2 * k, 2, stride=2), blok_konvolusi(4 * k, 2 * k)
        self.perbesar1, self.dec1 = nn.ConvTranspose2d(2 * k, k, 2, stride=2), blok_konvolusi(2 * k, k)
        self.keluaran = nn.Conv2d(k, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.ringkas(e1))
        e3 = self.enc3(self.ringkas(e2))
        d = self.dasar(self.ringkas(e3))
        d = self.dec3(torch.cat([self.perbesar3(d), e3], dim=1))   # skip connection
        d = self.dec2(torch.cat([self.perbesar2(d), e2], dim=1))
        d = self.dec1(torch.cat([self.perbesar1(d), e1], dim=1))
        return self.keluaran(d)


def dice_batch(tebakan, jawaban):
    irisan = (tebakan * jawaban).sum(dim=(1, 2, 3))
    return (2 * irisan + 1e-6) / (tebakan.sum(dim=(1, 2, 3)) + jawaban.sum(dim=(1, 2, 3)) + 1e-6)


def iou_batch(tebakan, jawaban):
    irisan = (tebakan * jawaban).sum(dim=(1, 2, 3))
    gabungan = tebakan.sum(dim=(1, 2, 3)) + jawaban.sum(dim=(1, 2, 3)) - irisan
    return (irisan + 1e-6) / (gabungan + 1e-6)


def latih(model, indeks, augmentasi=None, ukuran_batch=16):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    for epoch in range(JUMLAH_EPOCH):
        model.train()
        acak = indeks[torch.randperm(len(indeks))]
        for awal in range(0, len(acak), ukuran_batch):
            batch = acak[awal : awal + ukuran_batch]
            citra, jawaban = X[batch], Y[batch]
            if augmentasi is not None:
                citra, jawaban = terapkan_augmentasi(augmentasi, citra, jawaban)
            citra, jawaban = citra.to(device), jawaban.to(device)
            skor = model(citra)                                                    # 1. tebak
            loss = F.binary_cross_entropy_with_logits(skor, jawaban) \
                   + (1 - dice_batch(torch.sigmoid(skor), jawaban).mean())         # 2. hitung salah
            optimizer.zero_grad()                                                  # 3. bersihkan
            loss.backward()                                                        # 4. cari arah
            optimizer.step()                                                       # 5. perbaiki


def prediksi(model, indeks):
    model.eval()
    with torch.no_grad():
        return torch.cat([(torch.sigmoid(model(X[indeks[i : i + 50]].to(device))) > 0.5).float().cpu()
                          for i in range(0, len(indeks), 50)])

---
## Bagian 6 — Menjalankan eksperimen

Setiap skenario dilatih **`JUMLAH_ULANGAN` kali**. Pada ulangan ke-*n*, **semua skenario memakai citra latih yang sama dan titik awal model yang sama** — jadi perbedaan hasil benar-benar berasal dari augmentasinya. Ini yang disebut **perbandingan yang adil**.

Perkiraan waktu (200 citra, 40 epoch, 3 skenario × 3 ulangan): ±3–5 menit di GPU Colab, ±20–40 menit di laptop tanpa GPU. Sambil menunggu, mulai tulis README!

In [ ]:
hasil = {nama: {"dice": [], "iou": []} for nama in SKENARIO}
model_contoh = {}              # model dari ulangan pertama, dipakai untuk peta kesalahan
mulai_semua = time.time()

for ulangan in range(JUMLAH_ULANGAN):
    # Citra latih untuk ulangan ini — sama untuk semua skenario
    pilih = torch.randperm(len(kumpulan_latih), generator=torch.Generator().manual_seed(ulangan))[:JUMLAH_CITRA_LATIH]
    indeks_latih = kumpulan_latih[pilih]

    for nama, augmentasi in SKENARIO.items():
        mulai = time.time()
        torch.manual_seed(ulangan)                                   # titik awal model sama
        model = UNetMini().to(device)
        latih(model, indeks_latih, augmentasi)
        tebakan = prediksi(model, indeks_uji)
        hasil[nama]["dice"].append(dice_batch(tebakan, Y[indeks_uji]).mean().item())
        hasil[nama]["iou"].append(iou_batch(tebakan, Y[indeks_uji]).mean().item())
        if ulangan == 0:
            model_contoh[nama] = model
        print(f"Ulangan {ulangan + 1}/{JUMLAH_ULANGAN} | {nama:25s} | Dice = {hasil[nama]['dice'][-1]:.3f} "
              f"| IoU = {hasil[nama]['iou'][-1]:.3f} | {time.time() - mulai:.0f} detik")

print(f"\nSelesai dalam {(time.time() - mulai_semua) / 60:.1f} menit.")

---
## Bagian 7 — Ringkasan hasil

Sel di bawah mencetak **tabel siap tempel** untuk README dan menyimpan grafik perbandingan ke `hasil/perbandingan.png`. Angka ditulis sebagai **rata-rata ± simpangan baku** dari semua ulangan.

In [ ]:
baris = ["| Skenario | Dice (rata-rata ± sb) | IoU (rata-rata ± sb) | Dice tiap ulangan |", "|---|---|---|---|"]
for nama, h in hasil.items():
    d, i = np.array(h["dice"]), np.array(h["iou"])
    baris.append(f"| {nama} | {d.mean():.3f} ± {d.std():.3f} | {i.mean():.3f} ± {i.std():.3f} | {', '.join(f'{v:.3f}' for v in d)} |")
tabel = "\n".join(baris)

pengaturan = {"CEPAT": CEPAT, "JUMLAH_CITRA_LATIH": JUMLAH_CITRA_LATIH, "JUMLAH_EPOCH": JUMLAH_EPOCH,
              "JUMLAH_ULANGAN": JUMLAH_ULANGAN, "device": str(device), "skenario": {n: str(a) for n, a in SKENARIO.items()}}
(FOLDER_HASIL / "ringkasan.md").write_text(tabel + "\n")
(FOLDER_HASIL / "hasil.json").write_text(json.dumps({"pengaturan": pengaturan, "hasil": hasil}, indent=2))
print(tabel)
if CEPAT:
    print("\n⚠️  Ini hasil MODE CEPAT. Ganti CEPAT = False di Bagian 2 untuk hasil akhir.")

In [ ]:
nama_skenario = list(hasil)
rata = [np.mean(hasil[n]["dice"]) for n in nama_skenario]
sb = [np.std(hasil[n]["dice"]) for n in nama_skenario]

plt.figure(figsize=(max(6, 2.2 * len(nama_skenario)), 4.2))
batang = plt.bar(nama_skenario, rata, yerr=sb, capsize=6,
                 color=["tab:gray"] + ["tab:red"] * (len(nama_skenario) - 1))
for b, r in zip(batang, rata):
    plt.text(b.get_x() + b.get_width() / 2, r + 0.02, f"{r:.3f}", ha="center")
plt.ylabel("Dice pada 200 citra uji")
plt.ylim(0, 1)
plt.title(f"{JUMLAH_CITRA_LATIH} citra latih · {JUMLAH_EPOCH} epoch · {JUMLAH_ULANGAN} ulangan (garis = simpangan baku)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(FOLDER_HASIL / "perbandingan.png", dpi=120)
plt.show()

---
## Bagian 8 — Peta kesalahan

Lihat **di mana** model salah. **Hijau** = polip yang ditemukan dengan benar, **merah** = *false positive* (dikira polip, padahal bukan), **biru** = *false negative* (polip yang terlewat).

Baris yang ditampilkan: 2 citra uji yang **paling sulit** dan 2 yang **paling mudah** bagi model tanpa augmentasi. Bandingkan kolom-kolomnya: apakah augmentasi memperbaiki kasus sulit?

In [ ]:
def peta_kesalahan(tebakan, jawaban):
    gambar = np.zeros((*jawaban.shape, 3))
    gambar[(tebakan == 1) & (jawaban == 1)] = (0.1, 0.8, 0.3)
    gambar[(tebakan == 1) & (jawaban == 0)] = (0.9, 0.2, 0.2)
    gambar[(tebakan == 0) & (jawaban == 1)] = (0.2, 0.4, 1.0)
    return gambar


tebakan_semua = {nama: prediksi(m, indeks_uji) for nama, m in model_contoh.items()}
dice_dasar = dice_batch(tebakan_semua["Tanpa augmentasi"], Y[indeks_uji])
urut = dice_dasar.argsort()
dipilih = [urut[0], urut[1], urut[-2], urut[-1]]                # 2 tersulit, 2 termudah

kolom = 2 + len(model_contoh)
fig, axes = plt.subplots(4, kolom, figsize=(2.9 * kolom, 12))
for b, j in enumerate(dipilih):
    i = indeks_uji[j]
    axes[b, 0].imshow(X[i].permute(1, 2, 0).numpy()); axes[b, 0].set_title("Citra", fontsize=10)
    tampilkan_overlay(axes[b, 1], X[i], Y[i, 0], judul="Jawaban dokter")
    for k, (nama, teb) in enumerate(tebakan_semua.items()):
        d = dice_batch(teb[j : j + 1], Y[i : i + 1]).item()
        axes[b, 2 + k].imshow(peta_kesalahan(teb[j, 0].numpy(), Y[i, 0].numpy()))
        axes[b, 2 + k].set_title(f"{nama}\nDice = {d:.2f}", fontsize=10)
    for ax in axes[b]:
        ax.axis("off")
plt.tight_layout()
plt.savefig(FOLDER_HASIL / "peta_kesalahan.png", dpi=110)
plt.show()

---
## Bagian 9 — Selesai? Simpan ke repo kelompok

1. **Di laptop sendiri (VS Code / Jupyter):** simpan notebook (**Ctrl/Cmd + S**). File di `hasil/` sudah otomatis ada di folder repo.
2. **Di Google Colab:** jalankan sel di bawah untuk mengunduh `hasil.zip`, lalu **File → Download → Download .ipynb**. Ekstrak `hasil.zip` dan salin `hasil/` + notebook ke folder repo di laptopmu.
3. Lalu dari terminal di folder repo:
```bash
git pull
git add segmentasi_polip.ipynb hasil/
git commit -m "Jalankan eksperimen augmentasi: flip vs kecerahan"
git push
```
4. Tulis temuanmu di `README.md`, lalu commit & push lagi.

> Ingat: **commit wajib dari komputer masing-masing memakai perintah git**. Upload lewat halaman web GitHub tidak dihitung.

In [ ]:
# Hanya untuk Google Colab: bungkus folder hasil/ lalu unduh
try:
    from google.colab import files
    import shutil
    shutil.make_archive("hasil", "zip", ".", "hasil")
    files.download("hasil.zip")
except ImportError:
    print("Bukan di Colab — folder hasil/ sudah tersimpan di folder repo kamu.")